# Train CalorieSnap food model in Colab

In Colab, choose **Runtime → Change runtime type → T4 GPU** (if available), then run the cells in order. Dataset images are fetched into the temporary Colab runtime; they are not saved on your laptop. Colab runtimes are temporary, so download the exported model before disconnecting.

**Dataset size check:** the selected Hugging Face dataset currently has only 10 images. The next cell stops if any class has fewer than 5 images. This is a basic guard, not enough data for a dependable app model; target at least 50–100 diverse photos per dish.

In [ ]:
!pip -q install datasets tensorflow tensorflowjs scikit-learn

from collections import Counter
from datasets import load_dataset
import numpy as np
import tensorflow as tf

DATASET_ID = 'ychwodhrey/malaysian-cuisine'
ds = load_dataset(DATASET_ID, split='train')
print(ds)
print('Columns:', ds.column_names)
print('First row keys:', ds[0].keys())

In [ ]:
# Inspect label counts before spending time training.
label_column = 'label'
counts = Counter(str(row[label_column]).strip() for row in ds)
print('Images per class:', dict(counts))
MIN_IMAGES_PER_CLASS = 5
small_classes = {name: count for name, count in counts.items() if count < MIN_IMAGES_PER_CLASS}
if len(counts) < 2 or small_classes:
    raise ValueError(
        f'Not enough examples to train safely. Classes/counts: {dict(counts)}. '
        'Collect more photos per dish (aim for 50–100+) and add them to a Hugging Face imagefolder dataset.'
    )

# Show an example image and its label.
from IPython.display import display
print('Example label:', ds[0][label_column])
display(ds[0]['image'])

In [ ]:
# Decode/resize to arrays in Colab RAM. This dataset is small; for a large
# collection, switch to a streaming/batched tf.data input pipeline.
IMAGE_SIZE = (224, 224)
class_names = sorted(counts)
class_to_id = {name: i for i, name in enumerate(class_names)}
images, labels = [], []
for row in ds:
    image = row['image'].convert('RGB').resize(IMAGE_SIZE)
    images.append(np.asarray(image, dtype=np.float32))
    labels.append(class_to_id[str(row[label_column]).strip()])
x = np.stack(images) / 127.5 - 1.0
y = np.asarray(labels, dtype=np.int32)
from sklearn.model_selection import train_test_split
x_train, x_val, y_train, y_val = train_test_split(
    x, y, test_size=0.2, random_state=123, stratify=y
)
print(x_train.shape, x_val.shape, class_names)

In [ ]:
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.12),
])
base = tf.keras.applications.MobileNetV2(
    input_shape=(*IMAGE_SIZE, 3), include_top=False, weights='imagenet'
)
base.trainable = False
inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
z = augment(inputs)
z = base(z, training=False)
z = tf.keras.layers.GlobalAveragePooling2D()(z)
z = tf.keras.layers.Dropout(0.25)(z)
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(z)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=20,
          callbacks=[tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)])

In [ ]:
# Export files in the format the CalorieSnap frontend expects.
import json, os, subprocess, sys
from pathlib import Path
out = Path('/content/food-model')
out.mkdir(parents=True, exist_ok=True)
keras_path = '/content/food_classifier.keras'
model.save(keras_path)
(out / 'labels.json').write_text(json.dumps(class_names, indent=2))
subprocess.run([sys.executable, '-m', 'tensorflowjs.converters.converter',
                '--input_format=keras', keras_path, str(out)], check=True)
print('Exported:', list(out.iterdir()))

In [ ]:
# Download only the model export (not the photo dataset) to your computer.
from google.colab import files
import shutil
zip_path = '/content/food-model.zip'
shutil.make_archive('/content/food-model', 'zip', out)
files.download(zip_path)